# Notebook 06: Multi-Agent Pipeline — Target Assessment Report

**CABS AI Productivity Series**
Workshop: *LangChain, LangGraph & Local LLM Deployment: Building AI Agent Systems That Keep Your Data Safe*

---

## What This Notebook Covers

Notebooks 01–05 each feature a single agent or chain. In real research workflows, tasks are
too complex for one agent. A **multi-agent pipeline** breaks the problem into specialized roles:

```
User question
     │
     ▼
┌────────────────────┐
│  Literature Agent  │ ← Reads papers, summarises what is known
└────────────────────┘
     │ literature_summary
     ▼
┌────────────────────┐
│  Database Agent    │ ← Queries UniProt + ChEMBL for quantitative data
└────────────────────┘
     │ database_summary
     ▼
┌────────────────────┐
│  Report Writer     │ ← Synthesises both into a structured 1-page brief
└────────────────────┘
     │
     ▼
  Final Report
```

Each agent is a **node** in a **LangGraph StateGraph**. Agents share a common **state** dictionary
and pass their outputs forward to the next node.

---

## What You Will Learn

1. **LangGraph StateGraph** — define a graph of agent nodes with typed state
2. **Shared state** — how agents read from and write to a common data structure
3. **Conditional routing** — the graph decides which node to call next based on state
4. **Agent specialization** — each node has a focused job and the right tools for it
5. **End-to-end pipeline** — one call triggers the full multi-step workflow

---

## Data Privacy Note

This pipeline sends your target gene name to Google Gemini and to public APIs (UniProt, ChEMBL).
For unpublished targets, replace `ChatGoogleGenerativeAI` with `ChatOllama` to keep all LLM
inference local. The public API calls (UniProt, ChEMBL) remain the same — those databases are
designed to be queried publicly.


# Setup: Get Your Free Gemini API Key

1. Go to [aistudio.google.com](https://aistudio.google.com)
2. Sign in with your Google account → Accept Terms of Service
3. Left sidebar → **Get API Key** → **Create API key**
4. Copy the key (starts with `AIza...`)

No credit card needed.


In [ ]:
# ============================================================
# STEP 1: Install required packages
# ============================================================
# langchain              - framework for building LLM applications
# langchain-google-genai - connects LangChain to Google Gemini
# langgraph              - for building multi-agent state graphs
# requests               - for API calls to UniProt and ChEMBL

!pip install -q langchain langchain-google-genai langgraph requests
print("✅ Packages installed!")


In [ ]:
# ============================================================
# STEP 2: Enter your Gemini API key
# ============================================================

import getpass
import os

api_key = getpass.getpass("Paste your Gemini API key here: ")
os.environ["GOOGLE_API_KEY"] = api_key
print("✅ API key set!")


In [ ]:
# ============================================================
# STEP 3: Quick test — make sure Gemini is working
# ============================================================

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

response = llm.invoke("What is a target assessment in drug discovery? One sentence.")
print(response.content)
print("\n✅ Gemini is working!")


---
## Step 1: Define the Shared State

All agents in the pipeline read from and write to a shared `state` dictionary.
Think of it as a baton passed from one agent to the next — each one adds its results.

Using Python's `TypedDict`, we declare exactly what fields the state contains.


In [ ]:
# ============================================================
# STEP 4: Define the shared pipeline state
# ============================================================
# TypedDict tells Python (and LangGraph) what fields exist in the state.
# Every agent receives this full state and returns a partial update.

from typing import TypedDict, Optional


class TargetAssessmentState(TypedDict):
    target_gene: str              # INPUT: the gene/protein to assess (e.g. "EGFR")
    literature_summary: str       # OUTPUT of literature agent
    database_summary: str         # OUTPUT of database agent
    final_report: str             # OUTPUT of report writer


print("✅ State schema defined:")
print("   target_gene        — input from user")
print("   literature_summary — filled by Literature Agent")
print("   database_summary   — filled by Database Agent")
print("   final_report       — filled by Report Writer")


---
## Step 2: Build the Three Agent Nodes

Each node is a Python function that:
1. Receives the full state
2. Does its specialized job (LLM call, API calls, or both)
3. Returns a dict with only the fields it updated


In [ ]:
# ============================================================
# STEP 5: Node 1 — Literature Agent
# ============================================================
# This agent acts as a mini-RAG: it has a small built-in knowledge
# base of paper summaries and uses the LLM to synthesise them.
#
# In a production system, replace the hardcoded summaries with
# a real vector store (as built in Notebook 01).

import requests

LITERATURE_KB = {
    "EGFR": [
        "Lynch et al. 2004 (NEJM): Activating mutations in EGFR (exon 19 del, L858R) predict response "
        "to gefitinib in NSCLC. Patients with these mutations had 82% response rate vs 10% in wild-type.",
        "Mok et al. 2009 (NEJM, IPASS trial): Gefitinib superior to carboplatin/paclitaxel as first-line "
        "therapy in EGFR-mutant NSCLC (PFS HR 0.48). Established EGFR testing as standard of care.",
        "Kobayashi et al. 2005 (NEJM): T790M gatekeeper mutation in EGFR is the primary mechanism of "
        "acquired resistance to erlotinib/gefitinib, found in ~60% of resistant tumors.",
        "Soria et al. 2018 (NEJM, FLAURA): Osimertinib vs first-gen EGFR-TKI in EGFR-mutant NSCLC: "
        "PFS 18.9 vs 10.2 months (HR 0.46). Osimertinib is now standard first-line therapy.",
    ],
    "KRAS": [
        "Canon et al. 2019 (Nature): AMG 510 (sotorasib) covalently targets KRAS G12C via the switch-II "
        "pocket. First direct KRAS inhibitor to reach clinical trials.",
        "Skoulidis et al. 2021 (NEJM): Sotorasib showed 37.1% ORR in KRAS G12C NSCLC (CodeBreaK100). "
        "First FDA-approved KRAS inhibitor (May 2021).",
        "Hallin et al. 2020 (Cancer Discovery): MRTX849 (adagrasib) demonstrates broad activity across "
        "KRAS G12C tumor models including CNS penetration.",
        "Fell et al. 2020 (ACS Med Chem): Comprehensive review of KRAS drug discovery — covalent G12C "
        "inhibitors succeed where non-covalent pan-RAS approaches failed due to picomolar GDP affinity.",
    ],
    "BRCA1": [
        "Miki et al. 1994 (Science): BRCA1 cloned. Germline mutations confer ~80% lifetime breast cancer "
        "risk and ~40% ovarian cancer risk.",
        "Lord & Ashworth 2017 (Science): Synthetic lethality between BRCA1/2 mutations and PARP inhibition "
        "is the mechanistic basis for olaparib efficacy in BRCA-mutant cancers.",
        "Robson et al. 2017 (NEJM, OlympiAD): Olaparib significantly improved PFS vs chemotherapy in "
        "BRCA1/2-mutant metastatic breast cancer (7.0 vs 4.2 months, HR 0.58).",
        "Tutt et al. 2021 (NEJM, OlympiA): Adjuvant olaparib in high-risk HER2-negative BRCA1/2-mutant "
        "early breast cancer: 3-year distant DFS 87.5% vs 80.4% (HR 0.57).",
    ],
}

DEFAULT_LITERATURE = [
    "No curated literature summaries available for this target in the local knowledge base. "
    "The agent will rely on the LLM's training knowledge for this target.",
]


def literature_agent(state: TargetAssessmentState) -> dict:
    """Node 1: Summarise published literature for the target gene."""

    target = state["target_gene"]
    papers = LITERATURE_KB.get(target.upper(), DEFAULT_LITERATURE)

    paper_text = "\n".join(f"- {p}" for p in papers)

    prompt = f"""You are a scientific literature analyst. Based on the following paper summaries
about {target}, write a concise 3-4 sentence literature summary covering:
1. What is known about this target's biology and disease relevance
2. Key clinical or preclinical milestones
3. Major challenges or open questions

Paper summaries:
{paper_text}

Literature summary:"""

    response = llm.invoke(prompt)
    summary = response.content.strip()

    print(f"[Literature Agent] Summarised {len(papers)} papers for {target}.")
    return {"literature_summary": summary}


In [ ]:
# ============================================================
# STEP 6: Node 2 — Database Agent
# ============================================================
# This agent calls two public APIs:
#   1. UniProt — protein function, location, disease associations
#   2. ChEMBL  — known drugs and their clinical development stage
# It is an agent-with-tools (same pattern as Notebook 02).

from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool


@tool
def lookup_uniprot(gene_name: str) -> str:
    """Look up protein function, subcellular location, and disease associations in UniProt.
    Input: a gene symbol such as EGFR, KRAS, or BRCA1."""

    url = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query": f"(gene:{gene_name}) AND (organism_id:9606) AND (reviewed:true)",
        "format": "json",
        "size": 1,
        "fields": "accession,gene_names,protein_name,cc_function,cc_subcellular_location,cc_disease",
    }
    resp = requests.get(url, params=params, timeout=15)
    if resp.status_code != 200:
        return f"UniProt error: status {resp.status_code}"

    results = resp.json().get("results", [])
    if not results:
        return f"No UniProt entry found for '{gene_name}'."

    entry = results[0]
    accession = entry.get("primaryAccession", "N/A")

    prot_name = "N/A"
    rec = entry.get("proteinDescription", {}).get("recommendedName", {})
    if rec:
        prot_name = rec.get("fullName", {}).get("value", "N/A")

    gene_names = [g.get("geneName", {}).get("value", "") for g in entry.get("genes", [])]

    functions, locations, diseases = [], [], []
    for c in entry.get("comments", []):
        ctype = c.get("commentType")
        if ctype == "FUNCTION":
            functions += [t.get("value", "") for t in c.get("texts", [])]
        elif ctype == "SUBCELLULAR LOCATION":
            locations += [l.get("location", {}).get("value", "") for l in c.get("subcellularLocations", [])]
        elif ctype == "DISEASE":
            d = c.get("disease", {})
            if d:
                diseases.append(d.get("diseaseId", ""))

    return (
        f"UniProt {accession} | Gene: {', '.join(gene_names)} | Protein: {prot_name}\n"
        f"Function: {' '.join(functions)[:500] if functions else 'Not annotated'}\n"
        f"Location: {', '.join(locations) if locations else 'Not annotated'}\n"
        f"Diseases: {', '.join(diseases) if diseases else 'None annotated'}\n"
        f"URL: https://www.uniprot.org/uniprot/{accession}"
    )


@tool
def lookup_drugs_chembl(gene_name: str) -> str:
    """Search ChEMBL for approved or clinical-stage drugs that act on a given gene target.
    Input: a gene symbol such as EGFR, KRAS, or BRCA1.
    Returns up to 5 drugs with their clinical development phase."""

    url = "https://www.ebi.ac.uk/chembl/api/data/mechanism"
    params = {"format": "json", "limit": 100}
    resp = requests.get(url, params=params, timeout=15)
    if resp.status_code != 200:
        return f"ChEMBL error: status {resp.status_code}"

    # Filter mechanisms that mention the gene name
    mechanisms = resp.json().get("mechanisms", [])
    gene_upper = gene_name.upper()
    relevant = [
        m for m in mechanisms
        if gene_upper in (m.get("mechanism_of_action") or "").upper()
        or gene_upper in (m.get("target_chembl_id") or "").upper()
    ]

    if not relevant:
        # Try molecule name search as fallback
        mol_url = "https://www.ebi.ac.uk/chembl/api/data/molecule"
        mol_params = {"pref_name__icontains": gene_name, "format": "json", "limit": 5}
        mol_resp = requests.get(mol_url, params=mol_params, timeout=15)
        molecules = mol_resp.json().get("molecules", []) if mol_resp.status_code == 200 else []
        if not molecules:
            return f"No ChEMBL drug entries found targeting '{gene_name}'."
        lines = [f"Compounds with '{gene_name}' in name:"]
        for mol in molecules[:5]:
            lines.append(
                f"  • {mol.get('pref_name', 'N/A')} ({mol.get('molecule_chembl_id', 'N/A')}) "
                f"— Phase {mol.get('max_phase', 'N/A')}, approved: {mol.get('first_approval', 'N/A')}"
            )
        return "\n".join(lines)

    lines = [f"ChEMBL drugs/mechanisms targeting {gene_name} (showing up to 5):"]
    for m in relevant[:5]:
        lines.append(
            f"  • {m.get('molecule_chembl_id', 'N/A')} — "
            f"{m.get('mechanism_of_action', 'N/A')}"
        )
    return "\n".join(lines)


# Create the database agent with both tools
db_tools = [lookup_uniprot, lookup_drugs_chembl]
_db_agent = create_react_agent(model=llm, tools=db_tools)


def database_agent(state: TargetAssessmentState) -> dict:
    """Node 2: Query UniProt and ChEMBL for protein biology and drug landscape."""

    target = state["target_gene"]
    question = (
        f"Provide a concise summary of {target} covering: "
        f"(1) protein function and subcellular location from UniProt, "
        f"(2) known drugs or compounds in ChEMBL that target it, "
        f"(3) their highest clinical development phase. "
        f"Use the tools provided."
    )

    inputs = {"messages": [{"role": "user", "content": question}]}
    result = _db_agent.invoke(inputs)
    summary = result["messages"][-1].content.strip()

    print(f"[Database Agent] Retrieved UniProt + ChEMBL data for {target}.")
    return {"database_summary": summary}


In [ ]:
# ============================================================
# STEP 7: Node 3 — Report Writer
# ============================================================
# This node receives both summaries and synthesises a structured
# one-page target assessment brief — no tools needed, just the LLM.


def report_writer(state: TargetAssessmentState) -> dict:
    """Node 3: Synthesise all findings into a structured target brief."""

    target = state["target_gene"]

    prompt = f"""You are a senior drug discovery scientist writing an internal target assessment brief.

Target: {target}

LITERATURE EVIDENCE:
{state['literature_summary']}

DATABASE & DRUG LANDSCAPE:
{state['database_summary']}

Write a structured one-page target assessment brief with exactly these sections:

## Target: {target}

### 1. Biology & Disease Relevance
[2-3 sentences on mechanism and why this target matters]

### 2. Clinical Validation
[2-3 sentences on clinical evidence — approved drugs or trial results]

### 3. Drug Landscape
[Bullet list of approved/clinical drugs with their mechanism and phase]

### 4. Key Risks & Open Questions
[2-3 bullet points on scientific risks, resistance mechanisms, or gaps]

### 5. Recommendation
[1 sentence on whether to pursue this target and why]
"""

    response = llm.invoke(prompt)
    report = response.content.strip()

    print(f"[Report Writer] Target brief generated for {target}.")
    return {"final_report": report}


---
## Step 3: Build and Connect the Graph

Now we wire the three nodes together using `StateGraph`.
The flow is: **Literature → Database → Report Writer → END**.


In [ ]:
# ============================================================
# STEP 8: Build the LangGraph StateGraph
# ============================================================

from langgraph.graph import StateGraph, END


def build_pipeline():
    builder = StateGraph(TargetAssessmentState)

    # Register the three agent nodes
    builder.add_node("literature", literature_agent)
    builder.add_node("database", database_agent)
    builder.add_node("report", report_writer)

    # Wire the graph: entry → literature → database → report → END
    builder.set_entry_point("literature")
    builder.add_edge("literature", "database")
    builder.add_edge("database", "report")
    builder.add_edge("report", END)

    return builder.compile()


pipeline = build_pipeline()

print("✅ Multi-agent pipeline compiled!")
print("   Flow: literature_agent → database_agent → report_writer → END")


In [ ]:
# ============================================================
# STEP 9 (Optional): Visualise the graph
# ============================================================
# If pygraphviz or Mermaid is available in your environment,
# this will render a diagram of the pipeline.

try:
    from IPython.display import Image, display
    img_data = pipeline.get_graph().draw_mermaid_png()
    display(Image(img_data))
    print("✅ Graph visualisation rendered above.")
except Exception as e:
    print(f"Graph visualisation unavailable ({e})")
    print("Flow: literature_agent → database_agent → report_writer → END")


---
## Step 4: Run the Pipeline

One call triggers all three agents in sequence.


In [ ]:
# ============================================================
# STEP 10: Run the full pipeline for EGFR
# ============================================================

print("=" * 70)
print("  MULTI-AGENT TARGET ASSESSMENT PIPELINE")
print("=" * 70)
print()

result = pipeline.invoke({"target_gene": "EGFR"})

print()
print("=" * 70)
print("  PIPELINE COMPLETE — ALL THREE AGENTS FINISHED")
print("=" * 70)


In [ ]:
# ============================================================
# STEP 11: Display the final target assessment report
# ============================================================

print(result["final_report"])


In [ ]:
# ============================================================
# STEP 12: Inspect the intermediate summaries
# ============================================================
# Each agent's output is preserved in the final state.
# You can inspect any stage independently.

print("=== Literature Agent Output ===")
print(result["literature_summary"])
print()
print("=== Database Agent Output ===")
print(result["database_summary"])


In [ ]:
# ============================================================
# STEP 13: Run the pipeline for a second target — KRAS
# ============================================================

print("=" * 70)
print("  Running pipeline for KRAS")
print("=" * 70)
print()

result_kras = pipeline.invoke({"target_gene": "KRAS"})

print()
print(result_kras["final_report"])


In [ ]:
# ============================================================
# STEP 14: Run for any target you choose
# ============================================================
# Change the target_gene to any gene you are interested in.
# Targets without entries in our local literature KB will rely
# on the LLM's training knowledge + live UniProt/ChEMBL data.

MY_TARGET = "BRCA1"   # ← change this

result_custom = pipeline.invoke({"target_gene": MY_TARGET})
print(result_custom["final_report"])


---
## What You Built

You created a **three-node multi-agent pipeline** that:

1. **Literature Agent** — synthesises published evidence from an internal knowledge base
2. **Database Agent** — fetches live protein and drug data from UniProt and ChEMBL
3. **Report Writer** — combines both sources into a structured, scientist-readable brief

The key LangGraph concepts you used:

| Concept | Where it appeared |
|---|---|
| `TypedDict` state | Shared data structure passed between all nodes |
| `StateGraph` | Registered and connected the three nodes |
| `add_node` / `add_edge` | Defined the execution graph |
| `set_entry_point` | Designated the first node |
| `END` | Terminated the graph after the report is written |

---

## How to Extend This Pipeline

| Extension | How |
|---|---|
| Add a **Competitive Intelligence** node | New node that calls ClinicalTrials.gov (Notebook 04 tool) and inserts `competitor_summary` into state |
| Add a **human review** checkpoint | `interrupt_before=["report"]` in `builder.compile()` — pauses so a scientist can edit the summaries |
| Scale the literature KB | Replace the hardcoded dict with a real Chroma vector store (Notebook 01 pattern) |
| Run locally | Replace `ChatGoogleGenerativeAI` with `ChatOllama` (Notebook 03 pattern) — one line change |
| Process multiple targets in parallel | Loop over a list of genes and call `pipeline.invoke()` for each |

---

## Workshop Complete

You have now built:

| Notebook | What you built |
|---|---|
| 01 | RAG system for scientific literature Q&A |
| 02 | Agent with UniProt + AlphaFold tools |
| 03 | Local LLM deployment with Ollama |
| 04 | Drug discovery agent (ChEMBL + ClinicalTrials.gov) |
| 05 | Structured data extraction pipeline with Pydantic |
| **06** | **Multi-agent pipeline with shared state** |

Every pattern you learned — RAG, tool-calling, structured output, multi-agent graphs —
composes with the others. The one-line switch to a local LLM applies to all of them.
